# Ad Optimization Agent

A stateful AI agent built with **LangGraph** that reads real Q4 2021 campaign data from a CSV, allocates a $1,000 daily budget across four paid channels (Google Ads, Facebook, Instagram, YouTube), and logs every decision with rationale.

### Agent Loop
```
load_csv_data → agent_reasoning ──→ execute_tools → agent_reasoning → END
                               ↘ END (direct)
```

### Guardrails (auto-enforced in tools)
| Rule | Value |
|------|-------|
| Daily budget | $1,000 across 4 paid channels |
| Starting split | Equal — 25% ($250/day each) |
| Max daily change | ±20% per channel |
| Minimum floor | 5% (never 0%) |
| Max consecutive pause | 2 days |

**Primary metric:** Conversion Rate (CVR) | **Secondary:** ROI

## 1. Install Dependencies

In [7]:
%pip install langgraph langchain langchain_openai pydantic pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Imports, State & Global Budget State

In [8]:
import os
import operator
import pandas as pd
from typing import TypedDict, Annotated, List, Dict
from dotenv import load_dotenv

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

load_dotenv()

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Agent State
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    channel_data: Dict          # aggregated Q4 metrics per channel
    channel_allocations: Dict   # {channel: float pct}
    next_action: str

# Constants
PAID_CHANNELS = ["Google Ads", "Facebook", "Instagram", "YouTube"]
DAILY_BUDGET  = 1000.0

# Global mutable budget state (shared across all tool calls in one run)
BUDGET_STATE: Dict = {
    "allocations": {},
    "log":         [],
    "paused_days": {}
}

def initialize_budget():
    """Reset to equal 25% split before each agent run."""
    BUDGET_STATE["allocations"] = {ch: 25.0 for ch in PAID_CHANNELS}
    BUDGET_STATE["log"]         = []
    BUDGET_STATE["paused_days"] = {ch: 0 for ch in PAID_CHANNELS}

print("Imports OK.")

Imports OK.


## 3. Tools

Three tools the LLM can invoke. Guardrails are enforced inside each tool and recorded in the log.

| Tool | Purpose |
|------|---------|
| `allocate_budget` | Shift % to a channel (±20% cap, 5% floor) |
| `pause_channel` | Flag a channel as paused (max 2 consecutive days) |
| `request_new_creatives` | Request new ad assets for a channel |

In [9]:
@tool
def allocate_budget(channel: str, new_pct: float, reason: str) -> str:
    """
    Shift the daily budget percentage for `channel` to `new_pct`.
    Remaining % is distributed equally among the other three channels.
    Guardrails: max ±20% change; 5% minimum floor.
    """
    if channel not in PAID_CHANNELS:
        return f"Unknown channel '{channel}'. Valid: {PAID_CHANNELS}"

    current_pct = BUDGET_STATE["allocations"][channel]
    notes = []

    delta = new_pct - current_pct
    if delta > 20:
        new_pct = current_pct + 20
        notes.append(f"capped at +20% (requested {delta:+.1f}%)")
    elif delta < -20:
        new_pct = current_pct - 20
        notes.append(f"capped at -20% (requested {delta:+.1f}%)")

    if new_pct < 5.0:
        new_pct = 5.0
        notes.append("floored at 5%")

    others = [ch for ch in PAID_CHANNELS if ch != channel]
    per_other = round((100.0 - new_pct) / len(others), 4)
    for ch in others:
        BUDGET_STATE["allocations"][ch] = per_other
    BUDGET_STATE["allocations"][channel] = new_pct

    daily_amount = round(new_pct / 100 * DAILY_BUDGET, 2)
    BUDGET_STATE["log"].append({
        "action":          "allocate_budget",
        "channel":         channel,
        "from_pct":        current_pct,
        "to_pct":          new_pct,
        "daily_amount":    daily_amount,
        "reason":          reason,
        "guardrail_notes": "; ".join(notes) if notes else "none"
    })

    msg = (f"Budget for {channel}: {current_pct:.1f}% → {new_pct:.1f}% "
           f"(${daily_amount:.2f}/day). Reason: {reason}")
    if notes:
        msg += f" | Guardrails: {'; '.join(notes)}"
    return msg


@tool
def pause_channel(channel: str, reason: str) -> str:
    """
    Flag `channel` as paused for underperformance.
    A channel cannot be paused for more than 2 consecutive days.
    """
    if channel not in PAID_CHANNELS:
        return f"Unknown channel '{channel}'."

    consecutive = BUDGET_STATE["paused_days"].get(channel, 0)
    notes = []

    if consecutive >= 2:
        notes.append(f"pause blocked — already paused {consecutive} consecutive day(s)")
        status = "BLOCKED"
    else:
        BUDGET_STATE["paused_days"][channel] = consecutive + 1
        status = "PAUSED"

    BUDGET_STATE["log"].append({
        "action":           "pause_channel",
        "channel":          channel,
        "status":           status,
        "consecutive_days": BUDGET_STATE["paused_days"][channel],
        "reason":           reason,
        "guardrail_notes":  "; ".join(notes) if notes else "none"
    })

    msg = f"Channel {channel} {status}. Reason: {reason}"
    if notes:
        msg += f" | {'; '.join(notes)}"
    return msg


@tool
def request_new_creatives(channel: str, reason: str) -> str:
    """Request new creative assets for `channel` to improve engagement."""
    if channel not in PAID_CHANNELS:
        return f"Unknown channel '{channel}'."

    BUDGET_STATE["log"].append({
        "action":          "request_new_creatives",
        "channel":         channel,
        "reason":          reason,
        "guardrail_notes": "none"
    })
    return f"New creative assets requested for {channel}. Reason: {reason}"


ad_optimization_tools = [allocate_budget, pause_channel, request_new_creatives]
print("Tools defined.")

Tools defined.


## 4. Graph Nodes

In [10]:
def load_csv_data(state: AgentState) -> dict:
    """
    Node 1: reads marketing_campaign_dataset.csv, filters to Q4 2021
    (Oct-Dec) for the four paid channels, aggregates per-channel metrics,
    and primes the LLM with a performance summary.
    """
    print("--- LOADING CSV DATA ---")

    df = pd.read_csv("marketing_campaign_dataset.csv", parse_dates=["Date"])
    df['Acquisition_Cost'] = df['Acquisition_Cost'].str.replace('$', '').str.replace(',', '').astype(float)
    df = df[df["Channel_Used"].isin(PAID_CHANNELS)]
    df = df[
        (df["Date"].dt.year == 2021) &
        (df["Date"].dt.month.isin([10, 11, 12]))
    ]

    agg = df.groupby("Channel_Used").agg(
        avg_cvr=("Conversion_Rate",  "mean"),
        avg_roi=("ROI",              "mean"),
        avg_cpa=("Acquisition_Cost", "mean"),
        total_clicks=("Clicks",      "sum"),
        total_impressions=("Impressions", "sum"),
        row_count=("Campaign_ID",    "count")
    ).reset_index()
    agg["ctr"] = agg["total_clicks"] / agg["total_impressions"]

    channel_data = {}
    for _, row in agg.iterrows():
        channel_data[row["Channel_Used"]] = {
            "avg_cvr":   round(row["avg_cvr"],  4),
            "avg_roi":   round(row["avg_roi"],  4),
            "avg_cpa":   round(row["avg_cpa"],  2),
            "ctr":       round(row["ctr"],       4),
            "row_count": int(row["row_count"])
        }

    lines = ["Q4 2021 Channel Performance (paid channels only):\n"]
    for ch, m in channel_data.items():
        lines.append(
            f"  {ch}: CVR={m['avg_cvr']:.2%}  ROI={m['avg_roi']:.2f}  "
            f"CPA=${m['avg_cpa']:.2f}  CTR={m['ctr']:.2%}  (n={m['row_count']})"
        )

    lines.append("\nCurrent budget allocations (% of $1,000/day):")
    for ch, pct in BUDGET_STATE["allocations"].items():
        lines.append(f"  {ch}: {pct:.1f}% = ${pct/100*DAILY_BUDGET:.2f}")

    lines.append(
        "\nTask: analyse the channel performance above. "
        "Use tools to shift budget toward top performers (by CVR then ROI), "
        "pause poor performers, or request new creatives where engagement is low. "
        "After using all necessary tools, provide a concise final summary."
    )

    prompt = "\n".join(lines)
    print(prompt)

    return {
        "messages":            [HumanMessage(content=prompt)],
        "channel_data":        channel_data,
        "channel_allocations": dict(BUDGET_STATE["allocations"]),
        "next_action":         "start"
    }


def agent_reasoning(state: AgentState) -> dict:
    """Node 2: LLM analyses performance and selects a tool or finishes."""
    print("--- AGENT REASONING ---")

    llm_with_tools = llm.bind_tools(ad_optimization_tools)

    system_prompt = (
        "You are an expert Ad Optimization Agent managing a $1,000 daily budget "
        "across Google Ads, Facebook, Instagram, and YouTube. "
        "Primary metric: Conversion Rate (CVR). Secondary: ROI. "
        "Shift budget toward the highest CVR channels, pause channels with very low "
        "CVR and ROI, and request new creatives where engagement metrics are weak. "
        "Guardrails (+-20% cap, 5% floor, 2-day pause limit) are enforced automatically. "
        "After issuing all necessary tool calls, produce a concise final summary "
        "of your decisions and expected outcomes — do NOT call another tool."
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{messages}")
    ])

    response = (prompt | llm_with_tools).invoke(state)

    if response.tool_calls:
        return {"messages": [response], "next_action": "call_tool"}
    return {"messages": [response], "next_action": "FINISH"}


def execute_tools(state: AgentState) -> dict:
    """Node 3: runs the tool(s) selected by agent_reasoning."""
    print("--- EXECUTING TOOL ---")

    tool_map   = {t.name: t for t in ad_optimization_tools}
    tool_calls = state["messages"][-1].tool_calls
    results    = []

    for call in tool_calls:
        fn = tool_map.get(call["name"])
        if fn:
            try:
                result = fn.invoke(call["args"])
            except Exception as e:
                result = f"Error in {call['name']}: {e}"
        else:
            result = f"Tool '{call['name']}' not found."

        print(f"  [{call['name']}] -> {result}")
        results.append(ToolMessage(content=result, tool_call_id=call["id"]))

    return {
        "messages":            results,
        "channel_allocations": dict(BUDGET_STATE["allocations"])
    }


def decide_next_step(state: AgentState) -> str:
    """Router: tool call -> execute_tools; otherwise -> END."""
    return "call_tool" if state.get("next_action") == "call_tool" else "end"


print("Nodes defined.")

Nodes defined.


## 5. Build & Compile the LangGraph

In [11]:
workflow = StateGraph(AgentState)

workflow.add_node("load_csv_data",   load_csv_data)
workflow.add_node("agent_reasoning", agent_reasoning)
workflow.add_node("execute_tools",   execute_tools)

workflow.set_entry_point("load_csv_data")
workflow.add_edge("load_csv_data", "agent_reasoning")

workflow.add_conditional_edges(
    "agent_reasoning",
    decide_next_step,
    {"call_tool": "execute_tools", "end": END}
)

# After tools run, return to agent_reasoning for follow-up or final summary
workflow.add_edge("execute_tools", "agent_reasoning")

app = workflow.compile()
print("Graph compiled.")

Graph compiled.


## 6. Run the Agent

In [12]:
initialize_budget()

print("=" * 60)
print("STARTING AD OPTIMIZATION AGENT RUN")
print("=" * 60)

final_state = app.invoke(
    {
        "messages":            [],
        "channel_data":        {},
        "channel_allocations": {},
        "next_action":         "start"
    },
    config={"recursion_limit": 25}
)

print("\n" + "=" * 60)
print("AGENT FINAL SUMMARY")
print("=" * 60)
print(final_state["messages"][-1].content)

STARTING AD OPTIMIZATION AGENT RUN
--- LOADING CSV DATA ---
Q4 2021 Channel Performance (paid channels only):

  Facebook: CVR=8.03%  ROI=5.01  CPA=$12506.44  CTR=9.99%  (n=8256)
  Google Ads: CVR=8.02%  ROI=5.02  CPA=$12554.27  CTR=9.92%  (n=8431)
  Instagram: CVR=7.99%  ROI=5.00  CPA=$12550.95  CTR=9.93%  (n=8306)
  YouTube: CVR=8.01%  ROI=5.01  CPA=$12498.55  CTR=9.95%  (n=8481)

Current budget allocations (% of $1,000/day):
  Google Ads: 25.0% = $250.00
  Facebook: 25.0% = $250.00
  Instagram: 25.0% = $250.00
  YouTube: 25.0% = $250.00

Task: analyse the channel performance above. Use tools to shift budget toward top performers (by CVR then ROI), pause poor performers, or request new creatives where engagement is low. After using all necessary tools, provide a concise final summary.
--- AGENT REASONING ---
--- EXECUTING TOOL ---
  [allocate_budget] -> Budget for Facebook: 25.0% → 30.0% ($300.00/day). Reason: Highest CVR and ROI, increase budget allocation.
  [allocate_budget] -> Bu

## 7. Evaluation & Decision Log

Print the final budget split, the full decision log (with guardrail notes), and a per-channel performance snapshot.

In [13]:
print("=" * 60)
print("FINAL BUDGET ALLOCATIONS")
print("=" * 60)
total = 0.0
for ch, pct in BUDGET_STATE["allocations"].items():
    daily = pct / 100 * DAILY_BUDGET
    total += daily
    print(f"  {ch:15s}: {pct:6.2f}%  =  ${daily:7.2f}/day")
print(f"  {'TOTAL':15s}:          =  ${total:7.2f}/day")

print("\n" + "=" * 60)
print("DECISION LOG")
print("=" * 60)
if not BUDGET_STATE["log"]:
    print("  (no actions taken)")
for i, entry in enumerate(BUDGET_STATE["log"], 1):
    print(f"\n  [{i}] Action : {entry['action']}")
    for k, v in entry.items():
        if k != "action":
            print(f"      {k:20s}: {v}")

print("\n" + "=" * 60)
print("CHANNEL PERFORMANCE SNAPSHOT  (Q4 2021)")
print("=" * 60)
for ch, m in final_state.get("channel_data", {}).items():
    final_pct = BUDGET_STATE["allocations"].get(ch, 25.0)
    print(
        f"  {ch:15s}: CVR={m['avg_cvr']:.2%}  ROI={m['avg_roi']:.2f}  "
        f"CPA=${m['avg_cpa']:.2f}  CTR={m['ctr']:.2%}  -> {final_pct:.1f}% budget"
    )

FINAL BUDGET ALLOCATIONS
  Google Ads     :  26.67%  =  $ 266.67/day
  Facebook       :  26.67%  =  $ 266.67/day
  Instagram      :  20.00%  =  $ 200.00/day
  YouTube        :  26.67%  =  $ 266.67/day
  TOTAL          :          =  $1000.00/day

DECISION LOG

  [1] Action : allocate_budget
      channel             : Facebook
      from_pct            : 25.0
      to_pct              : 30.0
      daily_amount        : 300.0
      reason              : Highest CVR and ROI, increase budget allocation.
      guardrail_notes     : none

  [2] Action : allocate_budget
      channel             : Google Ads
      from_pct            : 23.3333
      to_pct              : 30.0
      daily_amount        : 300.0
      reason              : Second highest CVR and ROI, increase budget allocation.
      guardrail_notes     : none

  [3] Action : allocate_budget
      channel             : Instagram
      from_pct            : 23.3333
      to_pct              : 20.0
      daily_amount        : 200.